# 15. Lokális Térökonometria (Geographically Weighted Regression - GWR)

**Cél**: A térbeli heterogenitás modellezése Kőbányán. Szemben a SAR (NB13) modellel, amely globális $\rho$ együtthatót becsül, a GWR megengedi, hogy a magyarázó változók (pl. metrótól való távolság hatása) térben dinamikusan változzanak.
Például: Lehet, hogy Újhegyen a metró közelsége sokkal nagyobb felárat jelent, mint Óhegyen.

**Módszertan**: Az `mgwr` (Multiscale Geographically Weighted Regression) csomag használata. A paraméterfelületeket interaktív hőtérképen ábrázoljuk.

**Adatalap**: Az algoritmus stabilitása és az egyedi címek okozta szingularitás (lokális multikollinearitás) elkerülése érdekében **épület-szintű térbeli aggregációt** alkalmazunk azokon a lakásokon, amelyek azonos koordinátával rendelkeznek.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
try:
    from mgwr.gwr import GWR, MGWR
    from mgwr.sel_bw import Sel_BW
    MGWR_AVAILABLE = True
except ImportError:
    MGWR_AVAILABLE = False
    print("Figyelem: az 'mgwr' csomag nincs telepítve.")

df = load_szamitott_master()
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"GWR Mintaelemszám: {len(df_pontos)} db.")

GWR Mintaelemszám: 254 db.


### 1. GWR Sávszélesség (Bandwidth) Keresése és Modell Illesztése
A térbeli súlyozáshoz optimális távolságot (sávszélességet) az AICc kritérium minimalizálásával keressük.

In [2]:
features = ['korrigalt_alapterulet_nm', 'tavolsag_metro_halozati_m']
df_reg = df_pontos.dropna(subset=['log_nm_ar', 'geokodolt_lon', 'geokodolt_lat'] + features).copy()

# Tudományos megoldás a lokális multikollinearitás elkerülésére: 
# Térbeli aggregáció (egybeeső koordináták átlagolása épület/pont szinten)
df_agg = df_reg.groupby(['geokodolt_lon', 'geokodolt_lat'])[features + ['log_nm_ar']].mean().reset_index()
print(f"Eredeti hirdetések száma: {len(df_reg)} db.")
print(f"Térbeli aggregáció utáni egyedi pontok (épületek) száma: {len(df_agg)} db.")

coords = list(zip(df_agg['geokodolt_lon'], df_agg['geokodolt_lat']))
y_gwr = df_agg['log_nm_ar'].values.reshape((-1, 1))
X_gwr = df_agg[features].values

if MGWR_AVAILABLE:
    # Sávszélesség (Bandwidth) optimalizáció (kicsit időigényes lehet)
    print("GWR Sávszélesség optimalizálása folyamatban...")
    gwr_selector = Sel_BW(coords, y_gwr, X_gwr, fixed=False) # Adaptive bandwidth (KNN alapú)
    gwr_bw = gwr_selector.search()
    print(f"Optimális adaptív sávszélesség: {gwr_bw} legközelebbi szomszéd.")
    
    # Modell illesztése
    gwr_model = GWR(coords, y_gwr, X_gwr, gwr_bw, fixed=False)
    gwr_results = gwr_model.fit()
    
    print(f"GWR R²: {gwr_results.R2:.4f} (Adj. R²: {gwr_results.adj_R2:.4f})")
    print(f"GWR AICc: {gwr_results.aicc:.2f}")
    
    # Együtthatók kinyerése a dataframe-be
    # gwr_results.params egy (N, k) mátrix (k tartalmazza a konstanst is az első oszlopban)
    df_agg['gwr_const'] = gwr_results.params[:, 0]
    for i, col in enumerate(features):
        df_agg[f'gwr_{col}'] = gwr_results.params[:, i+1]
else:
    print("Az 'mgwr' csomag nélkül szimulált GWR paraméterfelületet generálunk.")
    df_agg['gwr_tavolsag_metro_halozati_m'] = -0.0001 + np.random.normal(0, 0.00005, len(df_agg))

Eredeti hirdetések száma: 254 db.
Térbeli aggregáció utáni egyedi pontok (épületek) száma: 124 db.
GWR Sávszélesség optimalizálása folyamatban...


Optimális adaptív sávszélesség: 121.0 legközelebbi szomszéd.
GWR R²: 0.0724 (Adj. R²: 0.0296)
GWR AICc: 4.73


### 2. A Metró Távolság Lokális Hatásának Térképes Vizualizációja
A hőtérképen (scatter_map) azt láthatjuk, hogy Kőbánya mely részein mennyire bünteti az árat a metrótól való távolság. (Ahol sötétebb kék, ott erősebb a negatív együttható, azaz "fájdalmasabb" a metrótól való távolság).

In [3]:
target_col = 'gwr_tavolsag_metro_halozati_m'

if target_col in df_agg.columns:
    fig = px.scatter_map(
        df_agg,
        lat='geokodolt_lat',
        lon='geokodolt_lon',
        color=target_col,
        size='korrigalt_alapterulet_nm',
        hover_data=[target_col, 'log_nm_ar'],
        color_continuous_scale='RdYlBu', # Red: gyenge hatás, Blue: erős negatív hatás
        map_style='carto-positron',
        zoom=12.2,
        title='GWR: A metrótávolság lokális regressziós együtthatója (Épület szinten aggregálva)'
    )
    fig.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
    fig.show()
else:
    print("A megjelenítéshez futtassa le a GWR modellt.")

### 3. Az Együtthatók Térbeli Eloszlása (Boxplot)
Megvizsgáljuk, hogy az egyes változók hatása mennyire ingadozik a kerületen belül.

In [4]:
if MGWR_AVAILABLE:
    # A lokális t-statisztikák és paraméterek eloszlása
    gwr_summary = pd.DataFrame(gwr_results.params, columns=['Konstans'] + features)
    
    fig = px.box(
        gwr_summary.melt(),
        x='variable',
        y='value',
        color='variable',
        title='GWR Regressziós Együtthatók Térbeli Szóródása (Heterogenitása)',
        template=PLOTLY_TEMPLATE
    )
    fig.update_layout(height=450, showlegend=False)
    fig.show()